# QA Tasks Consolidated Analysis

## Executive Summary

**Objective:** Consolidate and compare QA performance across all datasets by strategy type.

**Datasets Included:**
- ✅ **DocVQA_mini** (500 samples/phase, 8 phases) - Document Visual Question Answering
- ✅ **InfographicVQA_mini** (500 samples/phase, **11 phases including QA4**) - Infographic QA
- ✅ **dude_mini** (404 samples/phase, 8 phases) - Document Understanding (cleanest data)
- ✅ **chartqapro_mini** (494 samples/phase, 8 phases) - Chart QA (experimental, quality issues)

**QA Strategies:**
1. **QA1 (OCR+VLM):** Two-stage pipeline with OCR extraction + VLM answering
2. **QA2 (VLM Parse+QA):** VLM parses document, then answers questions
3. **QA3 (Direct-VQA):** Direct visual question answering without parsing
4. **QA4 (Special):** InfographicVQA-specific strategy with pre-extracted data

**Evaluation Metrics:**
- 🎯 **PRIMARY: GT in Pred** (Ground Truth in Prediction, higher is better, 0.0-1.0)
- **SECONDARY:** ANLS, Exact Match, Substring Match, Cosine Similarity

**Phases Mapping:**
- QA1a/b/c → OCR+VLM (simple/detailed/chain-of-thought prompts)
- QA2a/b/c → VLM-Parse+QA
- QA3a/b → Direct-VQA
- QA4a/b/c → Special Strategy (InfographicVQA only)

## Analysis Focus

1. **Strategy Comparison:** Which QA approach works best?
2. **Dataset Comparison:** Document vs Infographic vs Chart QA performance
3. **QA4 Special Analysis:** InfographicVQA unique strategy impact
4. **Model Ranking:** Best models per strategy
5. **Data Quality:** Handle empty predictions and errors

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import sys
from pathlib import Path
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

# Import evaluation metrics - GT in Pred is PRIMARY
from ocr_vs_vlm.metrics.evaluation_metrics import (
    compute_ground_truth_in_prediction,
    compute_anls,
    compute_exact_match
)

# Import embedding cache manager
from ocr_vs_vlm.metrics.embedding_cache import EmbeddingCacheManager, save_embeddings_for_phase

# Import model ordering
sys.path.insert(0, str(Path.cwd().parent))
from utils.model_order import MODEL_ORDER, sort_models, get_model_display_name

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 1000)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

print("✅ Libraries loaded successfully!")
print(f"📋 Model order: {', '.join(MODEL_ORDER)}")
print("\n🎯 PRIMARY METRIC: GT in Pred (Ground Truth in Prediction)")
print("   SECONDARY METRICS: ANLS, Exact Match, Substring Match, Cosine Similarity")

## 1. Setup

### Utility Functions

In [ ]:
# Utility functions for QA analysis
def parse_ground_truths(gt_value):
    """Parse ground truths from JSON string or list."""
    if pd.isna(gt_value):
        return []
    
    if isinstance(gt_value, list):
        return [str(x) for x in gt_value if pd.notna(x)]
    
    if isinstance(gt_value, str):
        try:
            parsed = json.loads(gt_value)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if x]
            return [str(parsed)]
        except:
            return [gt_value]
    
    return [str(gt_value)]

def is_valid_row(row, pred_col, err_col=None):
    """Check if row has valid prediction."""
    pred_value = row[pred_col]
    if pd.isna(pred_value) or str(pred_value).strip() == "":
        return False
    
    if err_col and err_col in row.index:
        if pd.notna(row[err_col]) and str(row[err_col]).strip() != "":
            return False
    
    return True

def get_phase_strategy(phase):
    """Map phase to strategy name."""
    if phase.startswith('QA1'):
        return 'QA1 (OCR+VLM)'
    elif phase.startswith('QA2'):
        return 'QA2 (VLM Parse+QA)'
    elif phase.startswith('QA3'):
        return 'QA3 (Direct-VQA)'
    elif phase.startswith('QA4'):
        return 'QA4 (Special)'
    else:
        return phase

def calculate_qa_metrics(prediction, ground_truths, phase, sample_id, model, emb_manager):
    """Calculate all QA metrics with GT in Pred as PRIMARY."""
    if pd.isna(prediction) or str(prediction).strip() == "":
        return {
            'gt_in_pred': 0.0,  # PRIMARY
            'anls': 0.0,
            'exact_match': 0.0,
            'substring_match': 0.0,
            'cosine_similarity': 0.0
        }
    
    pred_str = str(prediction)
    
    # PRIMARY METRIC
    gt_in_pred = compute_ground_truth_in_prediction(pred_str, ground_truths)
    
    # SECONDARY METRICS
    anls = compute_anls(pred_str, ground_truths)
    exact_match = compute_exact_match(pred_str, ground_truths)
    
    # Substring match (simple check)
    substring_match = float(any(gt.lower() in pred_str.lower() for gt in ground_truths))
    
    # Cosine similarity using embedding cache
    cosine_sim = 0.0
    if ground_truths:
        cosine_sim = emb_manager.compute_cosine_similarity(
            phase=phase,
            ground_truth=ground_truths[0],
            prediction=pred_str,
            sample_id=sample_id,
            model=model
        )
    
    return {
        'gt_in_pred': gt_in_pred,  # PRIMARY
        'anls': anls,
        'exact_match': exact_match,
        'substring_match': substring_match,
        'cosine_similarity': cosine_sim
    }

print("✅ Utility functions defined:")
print("   - parse_ground_truths(): Handle JSON ground truth formats")
print("   - is_valid_row(): Filter empty/error predictions")
print("   - get_phase_strategy(): Map phases to strategies")
print("   - calculate_qa_metrics(): Calculate all metrics (GT in Pred PRIMARY)")

### Dataset Configuration

In [ ]:
QA_DATASETS = {
    'DocVQA_mini': {
        'path': Path('../../2_clean/DocVQA_mini'),
        'task': 'document_qa',
        'task_display': 'Document QA',
        'phases': ['QA1a', 'QA1b', 'QA1c', 'QA2a', 'QA2b', 'QA2c', 'QA3a', 'QA3b']
    },
    'InfographicVQA_mini': {
        'path': Path('../../2_clean/InfographicVQA_mini'),
        'task': 'infographic_qa',
        'task_display': 'Infographic QA',
        'phases': ['QA1a', 'QA1b', 'QA1c', 'QA2a', 'QA2b', 'QA2c', 'QA3a', 'QA3b', 'QA4a', 'QA4b', 'QA4c']
    },
    'dude_mini': {
        'path': Path('../../2_clean/dude_mini'),
        'task': 'document_understanding',
        'task_display': 'Document Understanding',
        'phases': ['QA1a', 'QA1b', 'QA1c', 'QA2a', 'QA2b', 'QA2c', 'QA3a', 'QA3b']
    },
    'chartqapro_mini': {
        'path': Path('../../2_clean/chartqapro_mini'),
        'task': 'chart_qa',
        'task_display': 'Chart QA',
        'phases': ['QA1a', 'QA1b', 'QA1c', 'QA2a', 'QA2b', 'QA2c', 'QA3a', 'QA3b']
    }
}

EMBEDDINGS_DIR = Path('../../3_embeddings')

print("📁 Dataset Configuration:")
for name, config in QA_DATASETS.items():
    print(f"  - {name:25s}: {len(config['phases'])} phases, task={config['task']}")
print(f"\n📂 Embeddings directory: {EMBEDDINGS_DIR.resolve()}")